In [51]:
import os
import shutil
import json
from collections import Counter

import numpy as np
import pandas as pd
import yaml
from tqdm.contrib.concurrent import thread_map
from ultralytics import YOLO
from sklearn.model_selection import StratifiedGroupKFold

Example of project yaml:
```yaml
# Ultralytics YOLO 🚀, AGPL-3.0 license
# COCO128 dataset https://www.kaggle.com/ultralytics/coco128 (first 128 images from COCO train2017) by Ultralytics
# Example usage: yolo train data=coco128.yaml
# parent
# ├── ultralytics
# └── datasets
#     └── coco128  ← downloads here (7 MB)


# Train/val/test sets as 1) dir: path/to/imgs, 2) file: path/to/imgs.txt, or 3) list: [path/to/imgs1, path/to/imgs2, ..]
path: ../datasets/position_db_pose  # dataset root dir
train: images/train  # train images (relative to 'path') 128 images
val: images/val  # val images (relative to 'path') 128 images
test: images/test  # test images (optional)

kpt_shape: [1, 2]

# Classes
names:
  0: person
```

Example of annotaion txt file
```txt
0 0.726953125 0.3953125 0.27265625 0.20104166666666667 0.63359375 0.453125
0 0.288671875 0.35885416666666664 0.16484375 0.17395833333333333 0.36015625 0.41458333333333336
0 0.507421875 0.1359375 0.10703125 0.271875 0.49296875 0.25104166666666666
0 0.59765625 0.07864583333333333 0.090625 0.15729166666666666 0.58203125 0.10833333333333334
```

In [52]:
# base_path = "../../../db/object-detection"
base_path = "/mnt/pg-fs01/Current/DB/ObjectDetectionV3"
df = pd.read_csv(os.path.join(base_path, "annotation.csv"))
df.head()

,label,occluded,truncated,not_ground,occupied,occupied_by_person,ID,xtl,ytl,xbr,ybr,x,y,img_id,id
0,meeting_room_table,True,False,False,False,False,8.0,51.60,186.10,307.89,332.61,172.16,260.70,bdd640fb-0667-4ad1-9c80-317fa3b1799d,23b8c1e9-3924-46de-beb1-3b9046685257
1,bag,False,False,False,False,False,9.0,346.45,165.42,393.28,220.25,369.91,191.48,bdd640fb-0667-4ad1-9c80-317fa3b1799d,bd9c66b3-ad3c-4d6d-9a3d-1fa7bc8960a9
2,bag,True,False,False,False,False,10.0,329.10,133.89,369.63,170.60,350.15,150.37,bdd640fb-0667-4ad1-9c80-317fa3b1799d,972a8469-1641-4f82-8b9d-2434e465e150
3,laptop_closed,False,False,False,False,False,19.0,224.38,287.20,280.56,324.61,253.60,305.88,bdd640fb-0667-4ad1-9c80-317fa3b1799d,17fc695a-07a0-4a6e-8822-e8f36c031199
4,laptop_opened,False,False,False,False,False,18.0,197.42,191.27,249.80,228.00,224.38,210.00,bdd640fb-0667-4ad1-9c80-317fa3b1799d,9a1de644-815e-46d1-bb8f-aa1837f8a88b


In [53]:
df_imgs = pd.read_csv(os.path.join(base_path, "images_grouped.csv"))
df_imgs.head()

,id,name,width,height,batch,folder,group_id
0,bdd640fb-0667-4ad1-9c80-317fa3b1799d,Cam3_H243_discount_14.png,640.0,480.0,batch1_10.5,batch1,120287d6-9abf-49b8-ac00-72c72f9e383b
1,fc377a4c-4a15-444d-85e7-ce8a3a578a8e,Cam3_H243_discount_15.png,640.0,480.0,batch1_10.5,batch1,120287d6-9abf-49b8-ac00-72c72f9e383b
2,ff50bde4-3825-47b8-9cab-cc97663f1c97,Cam3_H243_discount_16.png,640.0,480.0,batch1_10.5,batch1,120287d6-9abf-49b8-ac00-72c72f9e383b
3,5cec4eb5-edd9-4831-9ca3-5cfb04fc6d82,Cam3_H243_discount_17.png,640.0,480.0,batch1_10.5,batch1,120287d6-9abf-49b8-ac00-72c72f9e383b
4,b92da22b-21df-406f-8a0b-3c3336d8393a,Cam3_H243_discount_18.png,640.0,480.0,batch1_10.5,batch1,120287d6-9abf-49b8-ac00-72c72f9e383b


Algo:
1. Read the annotaion and images;
2. Filter: remove: images without annotations, with errors, duplicates;
3. Select specific classes, merge if needed;
4. Map labels to class ids;
5. Convert to list of dict:
```python
{
    "id",
    "path",
    "objects": {
        "id",
        "cls",
        "bbox",
        "position"
    }
}
```
6. Split on train and val
7. Create YOLO dataset:
    1. label txt
    2. copy images
    3. create project.yaml file

In [54]:
# filter objects:
# - remove occupied by person
# - remove objects that occures in dataset < 1000 times

df = df[df["occupied_by_person"] == "False"]
popular_labels = df["label"].value_counts()[df["label"].value_counts() >= 1000].index.values
df = df[df["label"].isin(popular_labels)]

In [55]:
print("before", df_imgs.shape, df.shape)

# rm images without annotations
tmp_df = pd.merge(df, df_imgs, how="outer", left_on="img_id", right_on="id")
tmp_ids = tmp_df[tmp_df["id_x"].isna()]["id_y"].values
df_imgs = df_imgs[~df_imgs["id"].isin(tmp_ids)]
print("number of images without annotations:", len(tmp_ids))

# rm images with problems in annotations
tmp_ids = df[(df["xtl"].isna()) | (df["x"].isna())]["img_id"].unique()
df_imgs = df_imgs[~df_imgs["id"].isin(tmp_ids)]
df = df[~df["img_id"].isin(tmp_ids)]
print("number of images with 'problematic' annotations:", len(tmp_ids))

#rm duplicates
df1 = df_imgs.groupby(['name','batch', "folder", "id"]).size().reset_index()[['name','batch', "folder", "id"]]
duplicates = df1[df1.duplicated('name', keep=False)]
tmp_ids = duplicates.iloc[1::2]["id"].values
df_imgs = df_imgs[~df_imgs["id"].isin(tmp_ids)]
df = df[~df["img_id"].isin(tmp_ids)]
print("number of duplicates:", len(tmp_ids))

print("after", df_imgs.shape, df.shape)

before (1032, 7) (14893, 15)
number of images without annotations: 96
number of images with 'problematic' annotations: 30
number of duplicates: 10
after (896, 7) (14334, 15)


In [56]:
df_imgs["class"] = df_imgs["id"].apply(lambda x: df[df["img_id"] == x]["label"].mode()[0])
df_imgs["class"].value_counts(normalize=True)

class
office_chair     0.470982
desk             0.287946
laptop_opened    0.119420
monitor          0.089286
desk_FULL        0.032366
Name: proportion, dtype: float64

In [57]:
df["label"].value_counts(normalize=True)

label
office_chair     0.348751
desk             0.228548
desk_FULL        0.157249
laptop_opened    0.136877
monitor          0.128575
Name: proportion, dtype: float64

In [59]:
def dataframes2objects(df_annotations: pd.DataFrame, df_images: pd.DataFrame) -> tuple:
    label_counts = df["label"].value_counts().reset_index().to_dict(orient="records")
    labels = [i["label"] for i in label_counts]
    label_mapping = {idx: i for idx,i in enumerate(labels)}
    reverse_label_mapping = {i: idx for idx,i in enumerate(labels)}
    
    objects = []
    for _, row in df_images.iterrows():
        pth = os.path.join(row["batch"], row["folder"], row["name"])
        dets = []
        for _, row_det in df_annotations[df_annotations["img_id"] == row["id"]].iterrows():
            dets.append({
                "id": row_det["id"],
                "cls": reverse_label_mapping[row_det["label"]],
                "xtl": row_det["xtl"] / row["width"],
                "ytl": row_det["ytl"] / row["height"],
                "xbr": row_det["xbr"] / row["width"],
                "ybr": row_det["ybr"] / row["height"],
                "x": row_det["x"] / row["width"],
                "y": row_det["y"] / row["height"],
            })
        objects.append({
            "id": row["id"],
            "group_id": row["group_id"],
            "path": pth,
            "objects": dets,
        })
    
    return objects, label_mapping

In [60]:
def get_most_frequent_class(classes: list[str]) -> str:
    return Counter(classes).most_common(1)[0][0]

def split(objects: list, val_size: float, cl_ids: list[int], randon_seed: int, num_skip_valid_splits: int = 0) -> tuple:
    labels =sorted(cl_ids)

    image_ids = np.array([i['id'] for i in objects])
    group_ids = np.array([i['group_id'] for i in objects])
    most_frequent_classes = np.array([get_most_frequent_class([j["cls"] for j in i['objects']]) for i in objects])

    n_splits = round(1 / val_size)
    sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=randon_seed)
    current_try = 0
    # Add check for classes and groups
    for index, indices in enumerate(
        sgkf.split(image_ids, most_frequent_classes, groups=group_ids)
    ):
        print(f"Trying split # {index}")
        train_indexes, test_indexes = indices
        train_classes = [most_frequent_classes[i] for i in train_indexes]
        test_classes = [most_frequent_classes[i] for i in test_indexes]

        is_satisfactory_split = labels == sorted(np.unique(train_classes)) and labels == sorted(np.unique(test_classes))
        if is_satisfactory_split:
            if current_try >= num_skip_valid_splits:
                print("OK")
                break
            else:
                current_try += 1
        elif index == n_splits-1:
            print("not OK")
            break
    
    return image_ids[train_indexes], image_ids[test_indexes]

In [61]:
def create_project_file(file_pth: str, dataset_pth: str, label_mapping: dict) -> None:
    data = {
        "path": dataset_pth,
        "train": "images/train",
        "val": "images/val",
        "kpt_shape": [1, 2],
    }

    with open(file_pth, "wt") as f:
        yaml.dump(data, f, sort_keys=False, default_flow_style=None)
    
    with open(file_pth, "at") as f:
        yaml.dump({"names": label_mapping}, f, sort_keys=False)

In [62]:
def create_label_txt(args) -> None:
    pth, obj = args
    with open(os.path.join(pth, f"{obj['id']}.txt"), "wt") as f:
        for det in obj["objects"]:
            w = det["xbr"] - det["xtl"]
            h = det["ybr"] - det["ytl"]
            x_c = det["xtl"] + w / 2
            y_c = det["ytl"] + h / 2
            f.write(f"{det['cls']} {x_c} {y_c} {w} {h} {det['x']} {det['y']}\n")

def cp_img(args) -> None:
    base_pth, pth, obj = args
    shutil.copy(
        os.path.join(base_pth, obj["path"]),
        os.path.join(pth, f"{obj['id']}{os.path.splitext(obj['path'])[-1]}")
    )


def dummy_split(
        objects: list, val_size: float, cl_ids: list[int], randon_seed: int, num_skip_valid_splits: int = 0
) -> tuple:
    n = len(objects)
    n_val = int(n * val_size)
    idxs = np.arange(n)
    np.random.seed(randon_seed)
    np.random.shuffle(idxs)
    train_idxs = idxs[:-n_val]
    val_idxs = idxs[-n_val:]
    img_ids = np.array([i['id'] for i in objects])
    return img_ids[train_idxs], img_ids[val_idxs]


def create_yolo_dataset(
        objects: list, label_mapping: dict, base_imgs_pth: str, dataset_pth: str,
        val_size: float = 0.1, random_state: int = 42
) -> None:
    """_summary_

    Args:
        objects (list): list of objects. Object is an image with annotation:
        {
            "id": str,
            "group_id": str,
            "path": str,
            "objects": List[dict]: [
                {
                    "id": str,
                    "cls": int,
                    "xtl": float,
                    "ytl": float,
                    "xbr": float,
                    "ybr": float,
                    "x": float,
                    "y": float
                }
            ]
            
        }.
        label_mapping (dict): labels mapping, e.g. {0: desk, 1: chair}.
        base_imgs_pth (str): _description_
        dataset_pth (str): _description_
        val_size (float, optional): _description_. Defaults to 0.1.
        random_state (int, optional): _description_. Defaults to 42.
    """
    imgs_pth = os.path.join(dataset_pth, "images")
    labels_pth = os.path.join(dataset_pth, "labels")

    shutil.rmtree(dataset_pth, ignore_errors=True)
    os.makedirs(dataset_pth)
    os.makedirs(imgs_pth)
    os.makedirs(os.path.join(imgs_pth, "train"))
    os.makedirs(os.path.join(imgs_pth, "val"))
    os.makedirs(labels_pth)
    os.makedirs(os.path.join(labels_pth, "train"))
    os.makedirs(os.path.join(labels_pth, "val"))

    train_ids, val_ids = split(objects, val_size, list(label_mapping.keys()), random_state)
    print(len(train_ids), len(val_ids))
    with open(os.path.join(dataset_pth, "train_test_split.json"), "w") as f:
        json.dump({"train": train_ids.tolist(), "test": val_ids.tolist()}, f)

    train_objects = list(filter(lambda x: x["id"] in train_ids, objects))
    val_objects = list(filter(lambda x: x["id"] in val_ids, objects))

    thread_map(create_label_txt,
               zip([os.path.join(labels_pth, "train")] * len(train_ids), train_objects), desc="train labels")
    thread_map(create_label_txt,
               zip([os.path.join(labels_pth, "val")] * len(val_ids), val_objects), desc="val labels")

    thread_map(
        cp_img,
        zip([base_imgs_pth] * len(train_ids), [os.path.join(imgs_pth, "train")] * len(train_ids), train_objects),
        desc="train images", max_workers=4
    )
    thread_map(
        cp_img,
        zip([base_imgs_pth] * len(val_ids), [os.path.join(imgs_pth, "val")] * len(val_ids), val_objects),
        desc="val images", max_workers=4
    )

    create_project_file(os.path.join(dataset_pth, "project.yaml"), dataset_pth, label_mapping)

In [63]:
objects, label_mapping = dataframes2objects(df, df_imgs)

In [64]:

create_yolo_dataset(objects, label_mapping, base_path, os.path.join(os.getcwd(), "baseline"))

Trying split # 0
Trying split # 1
OK
758 138


train labels: 758it [00:00, 4007.52it/s]
val labels: 138it [00:00, 7360.86it/s]
train images: 758it [00:06, 117.88it/s]
val images: 138it [00:00, 180.10it/s]
